In [ ]:
!pip install transformers torch scikit-learn openpyxl

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
# LABELED FILES (for evaluation)
TITLE_LABELED = "/content/HN_AMD_Title_Only_Labeled_FinalFinal (1).xlsx"
TITLE_POST_LABELED = "/content/HN_AMD_Title_and_Post_Labeled_FinalFinal (1).xlsx"

# RAW FILES (for final prediction)
TITLE_RAW = "/content/HN_AMD_Title_Only (1).xlsx"
TITLE_POST_RAW = "/content/HN_AMD_Title_and_Post (1).xlsx"

In [ ]:
def get_embeddings(texts):
    embeddings = []
    finbert_preds = []

    for text in texts:
        inputs = tokenizer(
            str(text),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

        finbert_preds.append(pred)
        embeddings.append(logits.numpy().flatten())

    return np.array(embeddings), np.array(finbert_preds)

In [ ]:
def evaluate_models(filepath, text_column):

    df = pd.read_excel(filepath)

    texts = df[text_column].astype(str).tolist()
    labels = df["label"].values

    X, finbert_preds = get_embeddings(texts)

    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.2, random_state=42, stratify=labels
    )

    results = []

    # 1️⃣ FinBERT
    fb_test_preds = finbert_preds[:len(y_test)]

    results.append({
        "Model": "FinBERT",
        "Accuracy": accuracy_score(y_test, fb_test_preds),
        "Precision": precision_score(y_test, fb_test_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, fb_test_preds, average='weighted'),
        "F1": f1_score(y_test, fb_test_preds, average='weighted')
    })

    # 2️⃣ SVM
    svm = SVC(kernel='linear')
    svm.fit(X_train, y_train)
    svm_preds = svm.predict(X_test)

    results.append({
        "Model": "FinBERT+SVM",
        "Accuracy": accuracy_score(y_test, svm_preds),
        "Precision": precision_score(y_test, svm_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, svm_preds, average='weighted'),
        "F1": f1_score(y_test, svm_preds, average='weighted')
    })

    # 3️⃣ NBC
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    nb_preds = nb.predict(X_test)

    results.append({
        "Model": "FinBERT+NBC",
        "Accuracy": accuracy_score(y_test, nb_preds),
        "Precision": precision_score(y_test, nb_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, nb_preds, average='weighted'),
        "F1": f1_score(y_test, nb_preds, average='weighted')
    })

    # 4️⃣ Ensemble
    ensemble = VotingClassifier(
        estimators=[('svm', svm), ('nb', nb)],
        voting='hard'
    )
    ensemble.fit(X_train, y_train)
    ens_preds = ensemble.predict(X_test)

    results.append({
        "Model": "Ensemble",
        "Accuracy": accuracy_score(y_test, ens_preds),
        "Precision": precision_score(y_test, ens_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, ens_preds, average='weighted'),
        "F1": f1_score(y_test, ens_preds, average='weighted')
    })

    return pd.DataFrame(results)

In [ ]:
def evaluate_models(filepath, text_column):

    df = pd.read_excel(filepath)

    # ---------- CLEAN LABELS ----------
    df["label"] = df["label"].astype(str).str.lower().str.strip()

    label_mapping = {
        "negative": 0,
        "neutral": 1,
        "positive": 2,
        "0": 0,
        "1": 1,
        "2": 2
    }

    df["label"] = df["label"].map(label_mapping)

    # Check for unmapped values
    if df["label"].isnull().sum() > 0:
        print("⚠ Warning: Some labels could not be mapped:")
        print(df[df["label"].isnull()]["label"])
        return

    labels = df["label"].values
    texts = df[text_column].astype(str).tolist()

    X, finbert_preds = get_embeddings(texts)

    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.2, random_state=42, stratify=labels
    )

    results = []

    # 1️⃣ FinBERT
    fb_test_preds = finbert_preds[:len(y_test)]

    results.append({
        "Model": "FinBERT",
        "Accuracy": accuracy_score(y_test, fb_test_preds),
        "Precision": precision_score(y_test, fb_test_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, fb_test_preds, average='weighted'),
        "F1": f1_score(y_test, fb_test_preds, average='weighted')
    })

    # 2️⃣ SVM
    svm = SVC(kernel='linear')
    svm.fit(X_train, y_train)
    svm_preds = svm.predict(X_test)

    results.append({
        "Model": "FinBERT+SVM",
        "Accuracy": accuracy_score(y_test, svm_preds),
        "Precision": precision_score(y_test, svm_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, svm_preds, average='weighted'),
        "F1": f1_score(y_test, svm_preds, average='weighted')
    })

    # 3️⃣ NBC
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    nb_preds = nb.predict(X_test)

    results.append({
        "Model": "FinBERT+NBC",
        "Accuracy": accuracy_score(y_test, nb_preds),
        "Precision": precision_score(y_test, nb_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, nb_preds, average='weighted'),
        "F1": f1_score(y_test, nb_preds, average='weighted')
    })

    # 4️⃣ Ensemble
    ensemble = VotingClassifier(
        estimators=[('svm', svm), ('nb', nb)],
        voting='hard'
    )
    ensemble.fit(X_train, y_train)
    ens_preds = ensemble.predict(X_test)

    results.append({
        "Model": "Ensemble",
        "Accuracy": accuracy_score(y_test, ens_preds),
        "Precision": precision_score(y_test, ens_preds, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, ens_preds, average='weighted'),
        "F1": f1_score(y_test, ens_preds, average='weighted')
    })

    return pd.DataFrame(results)

In [ ]:
print("===== TITLE =====")
title_results = evaluate_models(TITLE_LABELED, "title")
print(title_results)

print("\n===== TITLE + POST =====")
title_post_results = evaluate_models(TITLE_POST_LABELED, "post")
print(title_post_results)

comparison = pd.concat([
    title_results.assign(Text="Title"),
    title_post_results.assign(Text="Title+Post")
])

comparison.to_excel("model_comparison.xlsx", index=False)

===== TITLE =====
         Model  Accuracy  Precision    Recall        F1
0      FinBERT  0.414286   0.616667  0.414286  0.410675
1  FinBERT+SVM  0.514286   0.483636  0.514286  0.459722
2  FinBERT+NBC  0.442857   0.419933  0.442857  0.382887
3     Ensemble  0.457143   0.384190  0.457143  0.356764

===== TITLE + POST =====
         Model  Accuracy  Precision    Recall        F1
0      FinBERT  0.705882   0.516242  0.705882  0.596349
1  FinBERT+SVM  0.720588   0.519247  0.720588  0.603570
2  FinBERT+NBC  0.779412   0.686765  0.779412  0.722882
3     Ensemble  0.779412   0.686765  0.779412  0.722882


In [ ]:
def train_best_model(filepath, text_column, model_type):

    df = pd.read_excel(filepath)
    texts = df[text_column].astype(str).tolist()
    labels = df["label"].values

    X, _ = get_embeddings(texts)

    if model_type == "SVM":
        model_final = SVC(kernel='linear')
    elif model_type == "NBC":
        model_final = GaussianNB()
    else:
        svm = SVC(kernel='linear')
        nb = GaussianNB()
        model_final = VotingClassifier(
            estimators=[('svm', svm), ('nb', nb)],
            voting='hard'
        )

    model_final.fit(X, labels)

    return model_final

In [ ]:
def predict_raw(raw_path, text_column, trained_model, output_name):

    df = pd.read_excel(raw_path)
    texts = df[text_column].astype(str).tolist()

    X, _ = get_embeddings(texts)

    df["predicted_sentiment"] = trained_model.predict(X)

    df.to_excel(output_name, index=False)

    print(f"Saved file: {output_name}")

In [ ]:
best_model = train_best_model(
    TITLE_LABELED,
    "title",
    model_type="FinBERT+SVM"  # CHANGE based on best result
)

predict_raw(
    TITLE_RAW,
    "title",
    best_model,
    "final_title_only_sentiment.xlsx"
)

Saved file: final_title_only_sentiment.xlsx


In [ ]:
best_model = train_best_model(
    TITLE_POST_LABELED,
    "post",
    model_type="Ensemble"  # CHANGE based on best result
)

predict_raw(
    TITLE_POST_RAW,
    "post",
    best_model,
    "final_title_post_sentiment.xlsx"
)

Saved file: final_title_post_sentiment.xlsx
